In [0]:
# ============================================================
# NOTEBOOK 03v2 — PREDICTIVE MODEL (sklearn)
# Vermont School - Early Warning System V2
#
# CAMBIO CLAVE: migración de SparkML a sklearn
# → elimina cache overflow
# → compatible con todos los modelos del curso
# → más rápido con 117 estudiantes
#
# TRAIN:   24-25 (T1+T2 features → risk_level target)
# PREDICT: 25-26 (T1+T2 features → predicted risk + T3 regression)
# ============================================================

# ── Rutas ──
TRUSTED = "/Volumes/workspace/vermont/trusted"
SILVER  = "/Volumes/workspace/vermont/silver"

TRAIN_FE_PATH   = f"{TRUSTED}/train_dataset_fe"
PREDICT_FE_PATH = f"{TRUSTED}/predict_dataset_fe"
MODEL_PATH      = f"{SILVER}/models"
PRED_OUTPUT     = f"{SILVER}/predictions_25_26"
DESCRIPTIVE     = f"{SILVER}/descriptive"

for path in [MODEL_PATH, PRED_OUTPUT]:
    dbutils.fs.mkdirs(path)

# ── Grupos de asignaturas ──
GROUPS = [
    'Science', 'I_and_S', 'Mathematics', 'English',
    'Lengua_Castellana', 'Mandarin', 'Financial_Maths',
    'ICT_STEM', 'Physical_Education', 'Research_Methodology'
]

# ── Features para clasificación ──
# T1 + T2 + deltas + variables de comportamiento
# ANTI-LEAKAGE: ninguna feature puede usar T3
FEATURES_CLASIF = (
    [f'{g}_T1' for g in GROUPS] +
    [f'{g}_T2' for g in GROUPS] +
    [f'{g}_delta' for g in GROUPS] +
    ['total_absences', 'absence_class', 'late', 'early_leave',
     'n_f1', 'n_f2', 'n_fault_types',
     'tendencia_general', 'avg_T1', 'avg_T2',
     'n_bajo_T1', 'n_bajo_T2', 'delta_materias_bajo',
     'dispersion_T2', 'ratio_ausencia_clase', 'indice_disciplinario',
     'min_nota_T2', 'min_nota_T1']
)

# ── Features para regresión ──
# Mismas features de clasificación como predictores
# Target: nota T3 por materia
FEATURES_REG = FEATURES_CLASIF.copy()

print("✓ Configuración cargada")
print(f"  Features clasificación definidas: {len(FEATURES_CLASIF)}")
print(f"  Features regresión definidas:     {len(FEATURES_REG)}")
print(f"  Grupos de asignaturas: {len(GROUPS)}")

In [0]:
# CELDA 2 — Carga de datos, preparación y validación anti-leakage

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

# ── Cargar desde Spark → pandas ──
df_train   = spark.read.parquet(TRAIN_FE_PATH).toPandas()
df_predict = spark.read.parquet(PREDICT_FE_PATH).toPandas()

print(f"✓ Train cargado:   {len(df_train)} estudiantes")
print(f"✓ Predict cargado: {len(df_predict)} estudiantes")

# ── Anti-leakage check ──
# Verificar que ninguna feature usa T3
t3_en_features = [f for f in FEATURES_CLASIF if 'T3' in f]
assert len(t3_en_features) == 0, \
    f"LEAKAGE DETECTADO: features con T3: {t3_en_features}"
print("\n✓ Anti-leakage check: ninguna feature de clasificación usa T3")

# ── Features disponibles ──
available_clasif = [f for f in FEATURES_CLASIF if f in df_train.columns]
missing_clasif   = [f for f in FEATURES_CLASIF if f not in df_train.columns]
print(f"\n✓ Features disponibles: {len(available_clasif)} de {len(FEATURES_CLASIF)}")
if missing_clasif:
    print(f"  Faltantes: {missing_clasif}")

# ── Preparar X e y para clasificación ──
X_raw = df_train[available_clasif].copy()
y     = df_train['risk_level'].copy()

# Imputación con media del train
# IMPORTANTE: guardamos el imputador para aplicarlo igual al predict
imputer = SimpleImputer(strategy='mean')
X = pd.DataFrame(
    imputer.fit_transform(X_raw),
    columns=available_clasif
)

# Codificar target
le = LabelEncoder()
y_enc = le.fit_transform(y)
classes = le.classes_

print(f"\n── Distribución de clases (train 24-25) ──")
for i, cls in enumerate(classes):
    n = np.sum(y_enc == i)
    pct = n / len(y_enc) * 100
    print(f"  {cls}: {n} estudiantes ({pct:.1f}%)")

# ── Class weights ──
n_total   = len(y_enc)
n_classes = len(classes)
class_weights_dict = {}
class_weights_arr  = np.ones(len(y_enc))
for i, cls in enumerate(classes):
    n_cls = np.sum(y_enc == i)
    w = n_total / (n_classes * n_cls)
    class_weights_dict[i] = round(w, 4)
    class_weights_arr[y_enc == i] = w

print(f"\n── Class weights (inversamente proporcionales) ──")
for i, cls in enumerate(classes):
    print(f"  {cls}: {class_weights_dict[i]}")
print(f"\n  → critical tiene peso {class_weights_dict[list(classes).index('critical')]:.2f}x")
print(f"    porque el costo de no detectarlo es mayor")

# ── Preparar X predict ──
X_predict_raw = df_predict[available_clasif].copy()
X_predict = pd.DataFrame(
    imputer.transform(X_predict_raw),  # usa media del TRAIN, no del predict
    columns=available_clasif
)

print(f"\n── Shapes finales ──")
print(f"  X train:   {X.shape}")
print(f"  y train:   {y_enc.shape}")
print(f"  X predict: {X_predict.shape}")

print(f"\n✓ Celda 2 completa — datos listos para modelado")
print(f"  Imputador ajustado con train → aplicado a predict")
print(f"  LabelEncoder: {dict(zip(range(len(classes)), classes))}")

In [0]:
%pip install lightgbm

In [0]:
# CELDA 3 — COMPARACIÓN DE MODELOS DE CLASIFICACIÓN
#
# Modelos: Logística, Árbol, Random Forest, GBT, LightGBM, Voting Ensemble
# Validación: Stratified 5-Fold CV (respeta proporción de clases)
# Métricas: PR-AUC, ROC-AUC, Balanced Acc, F1, Precision, Recall
#
# POR QUÉ STRATIFIED K-FOLD:
# Con solo 18 estudiantes críticos (15.4%), un fold aleatorio
# podría no tener ningún crítico → métricas engañosas.
# Stratified garantiza que cada fold mantiene la misma proporción.

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier,
                               GradientBoostingClassifier,
                               VotingClassifier)
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score, average_precision_score
)

# ── CV global — usado en Celda 4 también ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Definición de modelos ──
# Nota: GBT sklearn no tiene class_weight nativo
# LightGBM lo maneja con class_weight directamente

models = {
    'Logística': LogisticRegression(
        class_weight=class_weights_dict,
        max_iter=1000,
        random_state=42
    ),
    'Árbol Decisión': DecisionTreeClassifier(
        class_weight=class_weights_dict,
        max_depth=4,
        min_samples_leaf=3,
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        class_weight=class_weights_dict,
        n_estimators=150,
        max_depth=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ),
    'LightGBM': LGBMClassifier(
        class_weight=class_weights_dict,
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        verbose=-1
    ),
}

# ── Voting Ensemble: LR + RF + LightGBM ──
# Usamos LightGBM en vez de GBT por mejor manejo de desbalanceo
# soft voting → promedia probabilidades en vez de votos duros
models['Voting Ensemble'] = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(
            class_weight=class_weights_dict,
            max_iter=1000, random_state=42)),
        ('rf', RandomForestClassifier(
            class_weight=class_weights_dict,
            n_estimators=150, max_depth=5,
            random_state=42, n_jobs=-1)),
        ('lgbm', LGBMClassifier(
            class_weight=class_weights_dict,
            n_estimators=100, max_depth=4,
            learning_rate=0.1, random_state=42,
            verbose=-1))
    ],
    voting='soft'
)

print("── Modelos definidos ──")
for name in models:
    print(f"  {name}")

# ── Entrenamiento con Stratified 5-Fold CV ──
print("\nEntrenando con Stratified 5-Fold CV...")
print("(cada fold mantiene proporción crítico/recovery/no_risk)\n")

results = {}

for name, model in models.items():
    print(f"  → {name}...", end=" ")

    # cross_val_predict: cada estudiante es predicho por un modelo
    # que NO lo vio en entrenamiento → evaluación honesta
    y_pred = cross_val_predict(
        model, X, y_enc, cv=cv, method='predict'
    )
    y_proba = cross_val_predict(
        model, X, y_enc, cv=cv, method='predict_proba'
    )

    results[name] = {
        # Métricas globales
        'Accuracy':        round(accuracy_score(y_enc, y_pred), 4),
        'Balanced Acc':    round(balanced_accuracy_score(y_enc, y_pred), 4),
        'Precision macro': round(precision_score(y_enc, y_pred,
                               average='macro', zero_division=0), 4),
        'Recall macro':    round(recall_score(y_enc, y_pred,
                               average='macro', zero_division=0), 4),
        'F1 macro':        round(f1_score(y_enc, y_pred,
                               average='macro', zero_division=0), 4),
        'ROC-AUC (OvR)':   round(roc_auc_score(y_enc, y_proba,
                               multi_class='ovr', average='macro'), 4),
        'PR-AUC macro':    round(np.mean([
                               average_precision_score(
                                   (y_enc == i).astype(int),
                                   y_proba[:, i]
                               ) for i in range(len(classes))
                           ]), 4),
        # Recall por clase — crítico para Vermont
        # idx: critical=0, no_risk=1, recovery=2
        'Recall critical': round(recall_score(y_enc, y_pred,
                               labels=[0], average='macro',
                               zero_division=0), 4),
        'Recall recovery': round(recall_score(y_enc, y_pred,
                               labels=[2], average='macro',
                               zero_division=0), 4),
        'Recall no_risk':  round(recall_score(y_enc, y_pred,
                               labels=[1], average='macro',
                               zero_division=0), 4),
        # Guardar predicciones para visualizaciones en Celda 4
        '_y_pred':  y_pred,
        '_y_proba': y_proba,
    }

    print(f"✓ F1={results[name]['F1 macro']:.3f} | "
          f"Recall critical={results[name]['Recall critical']:.3f}")

# ── Tabla comparativa ──
display_cols = [
    'Accuracy', 'Balanced Acc', 'Precision macro', 'Recall macro',
    'F1 macro', 'ROC-AUC (OvR)', 'PR-AUC macro',
    'Recall critical', 'Recall recovery'
]

df_results = pd.DataFrame(
    {name: {k: v for k, v in m.items() if not k.startswith('_')}
     for name, m in results.items()}
).T[display_cols]

print("\n" + "="*100)
print("TABLA COMPARATIVA — MODELOS DE CLASIFICACIÓN")
print("Vermont School Early Warning System | Entrenado 24-25 | Stratified 5-Fold CV")
print("="*100)
print(df_results.to_string())
print("="*100)

# ── Análisis de resultados ──
best_f1      = df_results['F1 macro'].idxmax()
best_recall  = df_results['Recall critical'].idxmax()
best_auc     = df_results['ROC-AUC (OvR)'].idxmax()
best_pr_auc  = df_results['PR-AUC macro'].idxmax()

print(f"\n── Mejor modelo por métrica ──")
print(f"  F1 macro:        {best_f1:<20} ({df_results.loc[best_f1, 'F1 macro']:.3f})")
print(f"  Recall critical: {best_recall:<20} ({df_results.loc[best_recall, 'Recall critical']:.3f})")
print(f"  ROC-AUC:         {best_auc:<20} ({df_results.loc[best_auc, 'ROC-AUC (OvR)']:.3f})")
print(f"  PR-AUC:          {best_pr_auc:<20} ({df_results.loc[best_pr_auc, 'PR-AUC macro']:.3f})")

print(f"\n── Interpretación para Vermont ──")
print(f"  Recall critical es la métrica prioritaria:")
print(f"  → Falso negativo = estudiante crítico no detectado")
print(f"  → Costo institucional y personal muy alto")
print(f"  → Justifica aceptar algunas falsas alarmas")

# Modelo recomendado: mayor Recall critical con F1 aceptable
df_validos = df_results[df_results['F1 macro'] >= 0.50]
BEST_MODEL_NAME = df_validos['Recall critical'].idxmax()

print(f"\n── Modelo recomendado para producción ──")
print(f"  {BEST_MODEL_NAME}")
print(f"  Recall critical: {df_results.loc[BEST_MODEL_NAME, 'Recall critical']:.3f}")
print(f"  F1 macro:        {df_results.loc[BEST_MODEL_NAME, 'F1 macro']:.3f}")
print(f"  ROC-AUC:         {df_results.loc[BEST_MODEL_NAME, 'ROC-AUC (OvR)']:.3f}")

print(f"\n✓ Celda 3 completa")
print(f"  BEST_MODEL_NAME = '{BEST_MODEL_NAME}'")
print(f"  results dict disponible para Celda 4")

In [0]:
# ── Tabla comparativa ──
# Igual al ejemplo del profe en Sesión 3 (caso crédito)

display_cols = ['Accuracy', 'Balanced Acc', 'Precision macro',
                'Recall macro', 'F1 macro', 'ROC-AUC (OvR)',
                'PR-AUC macro', 'Recall critical', 'Recall recovery']

df_results = pd.DataFrame(
    {name: {k: v for k, v in metrics.items() if not k.startswith('_')}
     for name, metrics in results.items()}
).T[display_cols]

print("=" * 90)
print("TABLA COMPARATIVA — MODELOS DE CLASIFICACIÓN")
print("Vermont School Early Warning System — Entrenado 24-25, CV 5-Fold")
print("=" * 90)
print(df_results.to_string())
print("=" * 90)

# ── Análisis del mejor modelo ──
best_f1    = df_results['F1 macro'].idxmax()
best_recall = df_results['Recall critical'].idxmax()
best_auc   = df_results['ROC-AUC (OvR)'].idxmax()

print(f"\n── Análisis de resultados ──")
print(f"  Mejor F1 macro:        {best_f1} ({df_results.loc[best_f1, 'F1 macro']:.3f})")
print(f"  Mejor Recall critical: {best_recall} ({df_results.loc[best_recall, 'Recall critical']:.3f})")
print(f"  Mejor ROC-AUC:         {best_auc} ({df_results.loc[best_auc, 'ROC-AUC (OvR)']:.3f})")

print(f"\n── Interpretación para Vermont ──")
print(f"  → Recall critical es la métrica más importante:")
print(f"    un falso negativo = estudiante en riesgo no detectado")
print(f"    costo >> falso positivo (alerta innecesaria)")
print(f"  → El modelo seleccionado para producción será el de")
print(f"    mayor Recall critical con F1 macro aceptable (>0.60)")

# Identificar modelo recomendado
recomendado = df_results.loc[
    df_results['F1 macro'] >= 0.55,
    'Recall critical'
].idxmax()

print(f"\n  → MODELO RECOMENDADO: {recomendado}")
print(f"    Recall critical: {df_results.loc[recomendado, 'Recall critical']:.3f}")
print(f"    F1 macro:        {df_results.loc[recomendado, 'F1 macro']:.3f}")
print(f"    ROC-AUC:         {df_results.loc[recomendado, 'ROC-AUC (OvR)']:.3f}")

# Guardar nombre del mejor modelo para celdas siguientes
BEST_MODEL_NAME = recomendado
print(f"\n✓ Celda 3 completa — BEST_MODEL_NAME = '{BEST_MODEL_NAME}'")

In [0]:
# CELDA 4 — OPTIMIZACIÓN DE UMBRAL
# Comparación: Logística vs Random Forest
#
# HIPÓTESIS: RF con umbral ajustado puede superar
# a Logística con umbral por defecto (0.5) en Recall critical
#
# CRITERIOS DE SELECCIÓN DE UMBRAL:
# 1. Recall máximo    → detecta todos los críticos posibles
# 2. Youden Index     → maximiza TPR - FPR
# 3. F2-Score         → prioriza Recall, considera Precision

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_curve, fbeta_score
)
from sklearn.model_selection import cross_val_predict

# ── Modelos a comparar ──
modelos_comparar = {
    'Logística': LogisticRegression(
        class_weight=class_weights_dict,
        max_iter=1000,
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        class_weight=class_weights_dict,
        n_estimators=150,
        max_depth=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

idx_critical = list(classes).index('critical')
y_binary     = (y_enc == idx_critical).astype(int)
n_criticos   = y_binary.sum()

# ── Probabilidades out-of-fold ──
print("Calculando probabilidades out-of-fold...")
y_proba_lr = cross_val_predict(
    modelos_comparar['Logística'], X, y_enc,
    cv=cv, method='predict_proba'
)
y_proba_rf = cross_val_predict(
    modelos_comparar['Random Forest'], X, y_enc,
    cv=cv, method='predict_proba'
)

probas = {
    'Logística':     y_proba_lr[:, idx_critical],
    'Random Forest': y_proba_rf[:, idx_critical],
}

for nombre, p in probas.items():
    print(f"  {nombre}: rango {p.min():.3f} – {p.max():.3f}")

# ── Función evaluadora de umbrales ──
thresholds_to_eval = np.arange(0.10, 0.71, 0.05)

def evaluar_umbrales(proba_critical, y_enc, y_proba_full, nombre):
    resultados = []
    for thresh in thresholds_to_eval:
        y_pred_thresh = []
        for i in range(len(proba_critical)):
            if proba_critical[i] >= thresh:
                y_pred_thresh.append(idx_critical)
            else:
                proba_resto = y_proba_full[i].copy()
                proba_resto[idx_critical] = -1
                y_pred_thresh.append(np.argmax(proba_resto))
        y_pred_thresh = np.array(y_pred_thresh)

        recall_crit    = recall_score(y_enc, y_pred_thresh,
                             labels=[idx_critical], average='macro',
                             zero_division=0)
        precision_crit = precision_score(y_enc, y_pred_thresh,
                             labels=[idx_critical], average='macro',
                             zero_division=0)
        f1_macro       = f1_score(y_enc, y_pred_thresh,
                             average='macro', zero_division=0)
        f2_crit        = fbeta_score(y_binary,
                             (y_pred_thresh == idx_critical).astype(int),
                             beta=2, zero_division=0)
        tn = np.sum((y_pred_thresh != idx_critical) & (y_enc != idx_critical))
        fp = np.sum((y_pred_thresh == idx_critical) & (y_enc != idx_critical))
        specificity    = tn / (tn + fp) if (tn + fp) > 0 else 0
        youden         = recall_crit + specificity - 1
        tp             = int(np.sum(
                             (y_pred_thresh == idx_critical) &
                             (y_enc == idx_critical)))
        n_alertas      = int((y_pred_thresh == idx_critical).sum())

        resultados.append({
            'Modelo':             nombre,
            'Umbral':             round(thresh, 2),
            'Recall critical':    round(recall_crit, 3),
            'Precision critical': round(precision_crit, 3),
            'F2 critical':        round(f2_crit, 3),
            'Youden':             round(youden, 3),
            'F1 macro':           round(f1_macro, 3),
            'Specificity':        round(specificity, 3),
            'TP (detectados)':    tp,
            'N alertas':          n_alertas,
            'Falsas alarmas':     int(n_alertas - tp),
        })
    return pd.DataFrame(resultados)

df_lr = evaluar_umbrales(probas['Logística'],
                          y_enc, y_proba_lr, 'Logística')
df_rf = evaluar_umbrales(probas['Random Forest'],
                          y_enc, y_proba_rf, 'Random Forest')

# ── Tablas por modelo ──
cols_tabla = ['Umbral','Recall critical','Precision critical',
              'F2 critical','Youden','F1 macro',
              'TP (detectados)','Falsas alarmas']
print("\n── Logística — umbrales ──")
print(df_lr[cols_tabla].to_string(index=False))
print("\n── Random Forest — umbrales ──")
print(df_rf[cols_tabla].to_string(index=False))

# ── Comparación por criterio ──
print(f"\n{'='*70}")
print("COMPARACIÓN DE CRITERIOS — Logística vs Random Forest")
print(f"{'='*70}")
print(f"Críticos reales: {n_criticos} de {len(y_enc)} estudiantes\n")

resumen = []
for nombre, df in [('Logística', df_lr), ('Random Forest', df_rf)]:
    row_recall = df.loc[df['Recall critical'].idxmax()]
    row_youden = df.loc[df['Youden'].idxmax()]
    row_f2     = df.loc[df['F2 critical'].idxmax()]
    print(f"  {nombre}")
    print(f"  {'─'*50}")
    for criterio, row in [('Recall máximo', row_recall),
                           ('Youden Index',  row_youden),
                           ('F2-Score',      row_f2)]:
        print(f"  {criterio:<16} → umbral={row['Umbral']} | "
              f"Recall={row['Recall critical']} | "
              f"Precision={row['Precision critical']} | "
              f"F2={row['F2 critical']} | "
              f"Detecta {row['TP (detectados)']} de {n_criticos} | "
              f"Falsas={row['Falsas alarmas']}")
        resumen.append({'Modelo': nombre, 'Criterio': criterio,
                        **row[['Umbral','Recall critical',
                               'Precision critical','F2 critical',
                               'Youden','TP (detectados)',
                               'Falsas alarmas']].to_dict()})
    print()

df_resumen = pd.DataFrame(resumen)

# ── Ganador por criterio ──
print(f"{'='*70}")
print("GANADOR POR CRITERIO")
print(f"{'='*70}")
for criterio in ['Recall máximo', 'Youden Index', 'F2-Score']:
    sub     = df_resumen[df_resumen['Criterio'] == criterio]
    ganador = sub.loc[sub['Recall critical'].idxmax()]
    print(f"  {criterio:<16} → {ganador['Modelo']:<15} "
          f"(umbral={ganador['Umbral']}, "
          f"Recall={ganador['Recall critical']}, "
          f"Falsas={ganador['Falsas alarmas']})")

# ── Decisión final ──
mejor_lr = df_lr.loc[df_lr['F2 critical'].idxmax()]
mejor_rf = df_rf.loc[df_rf['F2 critical'].idxmax()]

if mejor_rf['Recall critical'] >= mejor_lr['Recall critical']:
    MODELO_PRODUCCION = 'Random Forest'
    UMBRAL_PRODUCCION = mejor_rf['Umbral']
    print(f"\n✓ HIPÓTESIS CONFIRMADA: RF con umbral {UMBRAL_PRODUCCION}")
    print(f"  supera a Logística en Recall critical")
else:
    MODELO_PRODUCCION = 'Logística'
    UMBRAL_PRODUCCION = mejor_lr['Umbral']
    print(f"\n✓ HIPÓTESIS RECHAZADA: Logística sigue siendo mejor")

print(f"\n── Modelo de producción Vermont ──")
print(f"  Modelo:  {MODELO_PRODUCCION}")
print(f"  Umbral:  {UMBRAL_PRODUCCION}")
print(f"  Detecta: {int(mejor_rf['TP (detectados)'])} de {n_criticos} críticos")
print(f"  Falsas:  {int(mejor_rf['Falsas alarmas'])} estudiantes")

# ── Visualizaciones ──
colores = {'Logística': '#185FA5', 'Random Forest': '#A32D2D'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    'Optimización de Umbral — Logística vs Random Forest\n'
    'Vermont Early Warning System | Clase Critical',
    fontsize=13, fontweight='bold'
)

# Plot 1: Recall critical vs Umbral
ax = axes[0, 0]
for nombre, df in [('Logística', df_lr), ('Random Forest', df_rf)]:
    ax.plot(df['Umbral'], df['Recall critical'], 'o-',
            color=colores[nombre], label=nombre, linewidth=2)
ax.axvline(x=UMBRAL_PRODUCCION, color='#639922', linestyle='--',
           label=f'Umbral óptimo RF ({UMBRAL_PRODUCCION})', alpha=0.8)
ax.set_xlabel('Umbral')
ax.set_ylabel('Recall critical')
ax.set_title('Recall critical vs Umbral')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 2: F2 vs Umbral
ax = axes[0, 1]
for nombre, df in [('Logística', df_lr), ('Random Forest', df_rf)]:
    ax.plot(df['Umbral'], df['F2 critical'], 's-',
            color=colores[nombre], label=nombre, linewidth=2)
ax.axvline(x=UMBRAL_PRODUCCION, color='#639922', linestyle='--',
           label=f'Umbral óptimo RF ({UMBRAL_PRODUCCION})', alpha=0.8)
ax.set_xlabel('Umbral')
ax.set_ylabel('F2-Score critical')
ax.set_title('F2-Score vs Umbral')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 3: Youden vs Umbral
ax = axes[1, 0]
for nombre, df in [('Logística', df_lr), ('Random Forest', df_rf)]:
    ax.plot(df['Umbral'], df['Youden'], 'D-',
            color=colores[nombre], label=nombre, linewidth=2)
ax.axvline(x=UMBRAL_PRODUCCION, color='#639922', linestyle='--',
           label=f'Umbral óptimo RF ({UMBRAL_PRODUCCION})', alpha=0.8)
ax.set_xlabel('Umbral')
ax.set_ylabel('Youden = TPR - FPR')
ax.set_title('Youden Index vs Umbral')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 4: Tradeoff TP vs Falsas alarmas
ax = axes[1, 1]
for nombre, df in [('Logística', df_lr), ('Random Forest', df_rf)]:
    ax.scatter(df['Falsas alarmas'], df['TP (detectados)'],
               color=colores[nombre], s=60, label=nombre, zorder=5)
    for _, row in df[df['Umbral'].isin([0.15, 0.25, 0.40, 0.50])].iterrows():
        ax.annotate(f"{row['Umbral']}",
                    (row['Falsas alarmas'], row['TP (detectados)']),
                    textcoords="offset points", xytext=(5, 3),
                    fontsize=7, color=colores[nombre])
ax.axhline(y=n_criticos, color='gray', linestyle='--',
           label=f'Detectar todos ({n_criticos})', alpha=0.5)
ax.set_xlabel('Falsas alarmas')
ax.set_ylabel('Críticos detectados (TP)')
ax.set_title('Tradeoff: Detección vs Falsas alarmas')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
display(fig)
plt.close()

# ── Curva ROC comparativa: Logística vs Random Forest ──
from sklearn.metrics import roc_curve, auc

print("\n" + "="*60)
print("CURVA ROC — Logística vs Random Forest")
print("Clase critical (one-vs-rest)")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'Curva ROC — Comparación Logística vs Random Forest\n'
    'Vermont EWS | CV 5-Fold | Clase Critical (one-vs-rest)',
    fontsize=12, fontweight='bold'
)

# ── Plot 1: ROC clase critical ──
ax = axes[0]
auc_scores_roc = {}
for nombre, y_proba, color in [
    ('Logística',     y_proba_lr, '#185FA5'),
    ('Random Forest', y_proba_rf, '#A32D2D'),
]:
    fpr, tpr, thresholds = roc_curve(
        y_binary, y_proba[:, idx_critical]
    )
    auc_score = auc(fpr, tpr)
    auc_scores_roc[nombre] = auc_score
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{nombre} (AUC = {auc_score:.3f})')

    # Marcar el punto correspondiente al umbral de producción
    if nombre == 'Random Forest':
        # Encontrar el punto más cercano al umbral 0.30
        idx_thresh = np.argmin(np.abs(thresholds - UMBRAL_PRODUCCION))
        ax.scatter(fpr[idx_thresh], tpr[idx_thresh],
                   color=color, s=120, zorder=5,
                   marker='*', label=f'RF umbral {UMBRAL_PRODUCCION}')
        ax.annotate(f'umbral={UMBRAL_PRODUCCION}\nTPR={tpr[idx_thresh]:.2f}',
                    (fpr[idx_thresh], tpr[idx_thresh]),
                    textcoords="offset points",
                    xytext=(10, -20), fontsize=8, color=color)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5,
        label='Clasificador aleatorio')
ax.fill_between(fpr, tpr, alpha=0.05, color='#A32D2D')
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR = Recall)')
ax.set_title('ROC — Clase Critical')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── Plot 2: ROC todas las clases (macro) ──
ax = axes[1]
from sklearn.preprocessing import label_binarize

y_bin = label_binarize(y_enc, classes=[0, 1, 2])
colores_clase = {
    'critical': '#A32D2D',
    'no_risk':  '#639922',
    'recovery': '#BA7517'
}

for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba_rf[:, i])
    auc_score   = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colores_clase[cls], linewidth=2,
            label=f'{cls} (AUC={auc_score:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5,
        label='Aleatorio')
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title('ROC por clase — Random Forest\n(one-vs-rest)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
display(fig)
plt.close()

print(f"\n── AUC Logística vs Random Forest (clase critical) ──")
for nombre, score in auc_scores_roc.items():
    print(f"  {nombre}: {score:.4f}")
print(f"\n  → Mayor AUC = mejor poder discriminativo global")
print(f"  → El punto estrella (*) muestra RF con umbral {UMBRAL_PRODUCCION}")
print(f"    en la curva ROC — su posición justifica la selección")

print(f"\n✓ Celda 4 completa")
print(f"  MODELO_PRODUCCION = '{MODELO_PRODUCCION}'")
print(f"  UMBRAL_PRODUCCION = {UMBRAL_PRODUCCION}")

In [0]:
# CELDA 4b — ACTUALIZACIÓN UMBRAL DE PRODUCCIÓN
# Justificación: umbral 0.30 maximiza F1 macro (0.663)
# con Recall critical 0.833 y solo 11 falsas alarmas
# Balance operativo óptimo para Vermont

UMBRAL_PRODUCCION = 0.30

# Verificar métricas en la tabla que ya calculamos
row_seleccionada = df_rf[df_rf['Umbral'] == UMBRAL_PRODUCCION].iloc[0]

print("=" * 60)
print("UMBRAL DE PRODUCCIÓN SELECCIONADO")
print("=" * 60)
print(f"\n  Modelo:            Random Forest")
print(f"  Umbral:            {UMBRAL_PRODUCCION}")
print(f"  Recall critical:   {row_seleccionada['Recall critical']}")
print(f"  Precision critical:{row_seleccionada['Precision critical']}")
print(f"  F2 critical:       {row_seleccionada['F2 critical']}")
print(f"  Youden:            {row_seleccionada['Youden']}")
print(f"  F1 macro:          {row_seleccionada['F1 macro']}")
print(f"  Detecta:           {row_seleccionada['TP (detectados)']} de 18 críticos reales")
print(f"  Falsas alarmas:    {row_seleccionada['Falsas alarmas']} estudiantes")

print(f"\n── Justificación vs umbral 0.15 ──")
row_015 = df_rf[df_rf['Umbral'] == 0.15].iloc[0]
print(f"  Umbral 0.15: Recall=1.0  | Falsas=24 | F1={row_015['F1 macro']}")
print(f"  Umbral 0.30: Recall=0.833| Falsas=11 | F1={row_seleccionada['F1 macro']}")
print(f"  → Perdemos 3 críticos pero eliminamos 13 falsas alarmas")
print(f"  → F1 macro sube de {row_015['F1 macro']} a {row_seleccionada['F1 macro']}")
print(f"  → Operativamente más viable para Vermont")

# ── Curva ROC con umbral 0.30 marcado ──
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_bin = label_binarize(y_enc, classes=[0, 1, 2])
colores_clase = {
    'critical': '#A32D2D',
    'no_risk':  '#639922',
    'recovery': '#BA7517'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'Curva ROC — Comparación Logística vs Random Forest\n'
    f'Vermont EWS | CV 5-Fold | Umbral seleccionado: {UMBRAL_PRODUCCION}',
    fontsize=12, fontweight='bold'
)

# ── Plot 1: ROC clase critical con umbral 0.30 ──
ax = axes[0]
auc_scores_roc = {}
for nombre, y_proba, color in [
    ('Logística',     y_proba_lr, '#185FA5'),
    ('Random Forest', y_proba_rf, '#A32D2D'),
]:
    fpr, tpr, thresholds = roc_curve(
        y_binary, y_proba[:, idx_critical]
    )
    auc_score = auc(fpr, tpr)
    auc_scores_roc[nombre] = auc_score
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{nombre} (AUC = {auc_score:.3f})')

    if nombre == 'Random Forest':
        idx_thresh = np.argmin(np.abs(thresholds - UMBRAL_PRODUCCION))
        ax.scatter(fpr[idx_thresh], tpr[idx_thresh],
                   color=color, s=150, zorder=5,
                   marker='*',
                   label=f'RF umbral {UMBRAL_PRODUCCION} '
                         f'(TPR={tpr[idx_thresh]:.2f}, '
                         f'FPR={fpr[idx_thresh]:.2f})')
        ax.annotate(
            f'umbral={UMBRAL_PRODUCCION}\n'
            f'TPR={tpr[idx_thresh]:.2f} | FPR={fpr[idx_thresh]:.2f}',
            (fpr[idx_thresh], tpr[idx_thresh]),
            textcoords="offset points",
            xytext=(10, -25), fontsize=8, color=color
        )

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5,
        label='Clasificador aleatorio')
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR = Recall)')
ax.set_title('ROC — Clase Critical')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── Plot 2: ROC todas las clases RF ──
ax = axes[1]
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba_rf[:, i])
    auc_score   = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colores_clase[cls], linewidth=2,
            label=f'{cls} (AUC={auc_score:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5,
        label='Aleatorio')
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title('ROC por clase — Random Forest\n(one-vs-rest)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
display(fig)
plt.close()

print(f"\n── AUC por modelo (clase critical) ──")
for nombre, score in auc_scores_roc.items():
    print(f"  {nombre}: {score:.4f}")

print(f"\n✓ Celda 4b completa")
print(f"  UMBRAL_PRODUCCION = {UMBRAL_PRODUCCION} (confirmado)")
print(f"  Listo para Celda 5 — entrenamiento final")

In [0]:
# CELDA 5 — ENTRENAMIENTO FINAL Y OPTIMIZACIÓN
# Random Forest con umbral 0.30
#
# FLUJO:
# 1. Grid Search → hiperparámetros óptimos con Stratified CV
# 2. Summary Train vs Validation (overfitting check)
# 3. Modelo final entrenado con todos los 117 estudiantes
# 4. Evaluación final con umbral 0.30

import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (StratifiedKFold, GridSearchCV,
                                      cross_val_predict)
from sklearn.metrics import (
    f1_score, recall_score, classification_report, fbeta_score
)

# ── 1. Grid Search ──
print("=" * 60)
print("PASO 1 — Grid Search: hiperparámetros óptimos")
print("=" * 60)
print("Optimizando F1 macro con Stratified 5-Fold CV\n")

param_grid = {
    'n_estimators':     [100, 150, 200],
    'max_depth':        [3, 4, 5, 6],
    'min_samples_leaf': [2, 3, 5],
    'max_features':     ['sqrt', 'log2']
}

total_combos = (len(param_grid['n_estimators']) *
                len(param_grid['max_depth']) *
                len(param_grid['min_samples_leaf']) *
                len(param_grid['max_features']))
print(f"  Combinaciones: {total_combos} × 5 folds = {total_combos*5} fits")

rf_base = RandomForestClassifier(
    class_weight=class_weights_dict,
    random_state=42,
    n_jobs=-1
)

cv_gs = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=cv_gs,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=0,
    return_train_score=True
)

print("  Corriendo Grid Search...", end=" ")
grid_search.fit(X, y_enc)
print("✓")

# ── 2. Summary Train vs Validation ──
print("\n" + "=" * 60)
print("PASO 2 — Summary Train vs Validation (overfitting check)")
print("=" * 60)

cv_results = pd.DataFrame(grid_search.cv_results_)

# Mejores hiperparámetros
print(f"\n── Mejores hiperparámetros ──")
for k, v in grid_search.best_params_.items():
    print(f"  {k:<20}: {v}")
print(f"  {'Mejor F1 macro (CV)':<20}: {grid_search.best_score_:.4f}")

# Top 10
top10 = cv_results.nlargest(10, 'mean_test_score')[
    ['param_n_estimators', 'param_max_depth',
     'param_min_samples_leaf', 'param_max_features',
     'mean_train_score', 'mean_test_score', 'std_test_score']
].round(4).copy()
top10.columns = ['n_est', 'depth', 'min_leaf', 'max_feat',
                  'Train F1', 'Val F1', 'Val std']
top10['Gap'] = (top10['Train F1'] - top10['Val F1']).round(4)
top10['Status'] = top10['Gap'].apply(
    lambda g: '✓ OK' if g < 0.10 else ('⚠ Moderado' if g < 0.20 else '✗ Overfit')
)

print(f"\n── Top 10 combinaciones ──")
print(top10.to_string(index=False))

# Summary completo
print(f"\n── Summary completo: todas las {total_combos} combinaciones ──")
summary = cv_results[[
    'param_n_estimators', 'param_max_depth',
    'param_min_samples_leaf', 'param_max_features',
    'mean_train_score', 'mean_test_score', 'std_test_score'
]].copy()
summary.columns = ['n_est', 'depth', 'min_leaf', 'max_feat',
                    'Train F1', 'Val F1', 'Val std']
summary['Gap (Train-Val)'] = (summary['Train F1'] - summary['Val F1']).round(4)
summary['Status'] = summary['Gap (Train-Val)'].apply(
    lambda g: '✓ OK' if g < 0.10 else ('⚠ Moderado' if g < 0.20 else '✗ Overfit')
)
summary = summary.sort_values('Val F1', ascending=False).round(4)
print(summary.to_string(index=False))

# Estadísticas del gap
print(f"\n── Estadísticas del gap Train-Val ──")
print(f"  Gap promedio:  {summary['Gap (Train-Val)'].mean():.4f}")
print(f"  Gap mínimo:    {summary['Gap (Train-Val)'].min():.4f}")
print(f"  Gap máximo:    {summary['Gap (Train-Val)'].max():.4f}")
n_ok = (summary['Gap (Train-Val)'] < 0.10).sum()
print(f"  Sin overfitting (gap<0.10): {n_ok} de {total_combos}")

# Gap del mejor modelo
mejor_gap = summary.iloc[0]['Gap (Train-Val)']
print(f"\n── Gap del mejor modelo ──")
print(f"  Train F1: {summary.iloc[0]['Train F1']:.4f}")
print(f"  Val F1:   {summary.iloc[0]['Val F1']:.4f}")
print(f"  Gap:      {mejor_gap:.4f}")
if mejor_gap < 0.10:
    print(f"  → Sin overfitting significativo ✓")
elif mejor_gap < 0.20:
    print(f"  → Gap moderado: modelo generaliza razonablemente")
else:
    print(f"  → Posible overfitting — considerar más regularización")

# ── Visualizaciones Grid Search ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    'Análisis Grid Search — Random Forest\n'
    'Vermont EWS | Stratified 5-Fold CV | scoring=F1 macro',
    fontsize=12, fontweight='bold'
)

# Plot 1: Val F1 vs max_depth por n_estimators
ax = axes[0]
for n_est in param_grid['n_estimators']:
    mask = cv_results['param_n_estimators'] == n_est
    sub  = cv_results[mask].groupby('param_max_depth')[
        'mean_test_score'].mean().reset_index()
    ax.plot(sub['param_max_depth'], sub['mean_test_score'],
            'o-', linewidth=2, label=f'n_est={n_est}')
ax.set_xlabel('max_depth')
ax.set_ylabel('Val F1 macro (CV)')
ax.set_title('Val F1 vs max_depth')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 2: Val F1 vs min_samples_leaf por max_features
ax = axes[1]
for feat in param_grid['max_features']:
    mask = cv_results['param_max_features'] == feat
    sub  = cv_results[mask].groupby('param_min_samples_leaf')[
        'mean_test_score'].mean().reset_index()
    ax.plot(sub['param_min_samples_leaf'], sub['mean_test_score'],
            's-', linewidth=2, label=f'max_feat={feat}')
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('Val F1 macro (CV)')
ax.set_title('Val F1 vs min_samples_leaf')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 3: Train vs Val F1 (overfitting visual)
ax = axes[2]
ax.scatter(cv_results['mean_train_score'],
           cv_results['mean_test_score'],
           alpha=0.5, color='#185FA5', s=40, label='Combinaciones')
# Marcar el mejor
best_idx = cv_results['mean_test_score'].idxmax()
ax.scatter(cv_results.loc[best_idx, 'mean_train_score'],
           cv_results.loc[best_idx, 'mean_test_score'],
           color='#A32D2D', s=120, zorder=5,
           marker='*', label='Mejor combinación')
# Línea diagonal (Train = Val → sin overfitting)
lims = [
    min(cv_results['mean_train_score'].min(),
        cv_results['mean_test_score'].min()) - 0.02,
    max(cv_results['mean_train_score'].max(),
        cv_results['mean_test_score'].max()) + 0.02
]
ax.plot(lims, lims, 'k--', alpha=0.4, label='Train = Val (ideal)')
ax.set_xlabel('Train F1 macro')
ax.set_ylabel('Val F1 macro (CV)')
ax.set_title('Train vs Val F1\n(puntos sobre diagonal = overfitting)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
display(fig)
plt.close()

# ── 3. Modelo final ──
print("\n" + "=" * 60)
print("PASO 3 — Modelo final entrenado con todos los 117")
print("=" * 60)

rf_final = RandomForestClassifier(
    **grid_search.best_params_,
    class_weight=class_weights_dict,
    random_state=42,
    n_jobs=-1
)
rf_final.fit(X, y_enc)
print(f"  ✓ Modelo entrenado con {len(X)} estudiantes")
print(f"  ✓ Features:  {X.shape[1]}")
print(f"  ✓ Árboles:   {rf_final.n_estimators}")
print(f"  ✓ max_depth: {rf_final.max_depth}")

# ── 4. Evaluación final con umbral 0.30 ──
print("\n" + "=" * 60)
print(f"PASO 4 — Evaluación final con umbral {UMBRAL_PRODUCCION}")
print("=" * 60)

cv_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_proba_final = cross_val_predict(
    rf_final, X, y_enc,
    cv=cv_final,
    method='predict_proba'
)

# Aplicar umbral 0.30
y_pred_final = []
for i in range(len(y_proba_final)):
    if y_proba_final[i, idx_critical] >= UMBRAL_PRODUCCION:
        y_pred_final.append(idx_critical)
    else:
        proba_resto = y_proba_final[i].copy()
        proba_resto[idx_critical] = -1
        y_pred_final.append(np.argmax(proba_resto))
y_pred_final = np.array(y_pred_final)

# Métricas
f1_final     = f1_score(y_enc, y_pred_final,
                         average='macro', zero_division=0)
recall_final = recall_score(y_enc, y_pred_final,
                             labels=[idx_critical], average='macro',
                             zero_division=0)
f2_final     = fbeta_score(
    (y_enc == idx_critical).astype(int),
    (y_pred_final == idx_critical).astype(int),
    beta=2, zero_division=0
)
tp_final     = int(np.sum(
    (y_pred_final == idx_critical) & (y_enc == idx_critical)
))
falsas_final = int(np.sum(
    (y_pred_final == idx_critical) & (y_enc != idx_critical)
))

print(f"\n── Reporte de clasificación ──")
print(classification_report(
    y_enc, y_pred_final,
    target_names=classes, zero_division=0
))

print(f"── Métricas clave ──")
print(f"  F1 macro:          {f1_final:.4f}")
print(f"  Recall critical:   {recall_final:.4f}")
print(f"  F2 critical:       {f2_final:.4f}")
print(f"  Detecta:           {tp_final} de 18 críticos reales")
print(f"  Falsas alarmas:    {falsas_final} estudiantes")

# Comparación base vs optimizado
print(f"\n── Comparación: base vs optimizado (umbral 0.30) ──")
print(f"  {'Métrica':<22} {'Base RF (Celda 3)':<22} {'RF Optimizado':<22}")
print(f"  {'─'*66}")
print(f"  {'F1 macro':<22} {'0.5734':<22} {f1_final:.4f}")
print(f"  {'Recall critical':<22} {'0.3889':<22} {recall_final:.4f}")
print(f"  {'F2 critical':<22} {'—':<22} {f2_final:.4f}")
print(f"  {'Detecta críticos':<22} {'7 de 18':<22} {tp_final} de 18")
print(f"  {'Falsas alarmas':<22} {'—':<22} {falsas_final}")
print(f"  {'Umbral':<22} {'0.50 (default)':<22} {UMBRAL_PRODUCCION}")

print(f"\n✓ Celda 5 completa")
print(f"  rf_final listo para Celdas 6, 7 y 8")
print(f"  y_proba_final listo para visualizaciones")
print(f"  Umbral de producción: {UMBRAL_PRODUCCION}")

In [0]:
# CELDA 6 — APLICACIÓN SOBRE 25-26 Y CATEGORÍAS DE ALERTA
#
# Aplica rf_final sobre los 149 estudiantes de 2025-26
# Cruza predicción del modelo con realidad parcial (T3)
# Genera 4 categorías basadas en matriz de confusión:
#
#                    | T3 confirma riesgo | T3 no confirma |
# Modelo dice riesgo | 🔴 Confirmado      | 🔵 Teórico     |
# Modelo dice ok     | 🟠 Punto ciego     | 🟢 Sin riesgo  |
#
# "T3 confirma riesgo" = al menos 1 materia con acumulada < 4.0

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

print("=" * 60)
print("PASO 1 — Predicción sobre 25-26 (149 estudiantes)")
print("=" * 60)

# ── Aplicar modelo final sobre predict ──
y_proba_predict = rf_final.predict_proba(X_predict)
proba_critical_predict = y_proba_predict[:, idx_critical]

# Aplicar umbral 0.30
y_pred_predict = []
for i in range(len(y_proba_predict)):
    if y_proba_predict[i, idx_critical] >= UMBRAL_PRODUCCION:
        y_pred_predict.append(idx_critical)
    else:
        proba_resto = y_proba_predict[i].copy()
        proba_resto[idx_critical] = -1
        y_pred_predict.append(np.argmax(proba_resto))
y_pred_predict = np.array(y_pred_predict)

# Mapear índices a etiquetas
pred_labels = le.inverse_transform(y_pred_predict)
confianza   = np.max(y_proba_predict, axis=1)

print(f"\n── Distribución de predicciones (25-26) ──")
for cls in classes:
    n = (pred_labels == cls).sum()
    print(f"  {cls:<12}: {n:>3} estudiantes ({n/len(pred_labels)*100:.1f}%)")

print(f"\n── Confianza del modelo ──")
print(f"  Media:  {confianza.mean():.3f}")
print(f"  Mín:    {confianza.min():.3f}")
print(f"  Máx:    {confianza.max():.3f}")
print(f"  < 0.50: {(confianza < 0.50).sum()} estudiantes (baja confianza)")

# ── PASO 2: Calcular T3 parcial acumulada ──
print("\n" + "=" * 60)
print("PASO 2 — Calcular riesgo real con T3 parcial")
print("=" * 60)
print("Criterio: acumulada = T1×0.30 + T2×0.30 + T3×0.40")
print("T3 confirma riesgo = al menos 1 materia con acumulada < 4.0")

df_pred = df_predict.copy()

# Calcular acumulada por materia
materias_en_bajo = []
for idx_est in range(len(df_pred)):
    n_bajo = 0
    for g in GROUPS:
        t1 = f'{g}_T1'
        t2 = f'{g}_T2'
        t3 = f'{g}_T3'
        if all(c in df_pred.columns for c in [t1, t2, t3]):
            t1v = df_pred.iloc[idx_est][t1]
            t2v = df_pred.iloc[idx_est][t2]
            t3v = df_pred.iloc[idx_est][t3]
            if not any(pd.isna([t1v, t2v, t3v])):
                acum = t1v * 0.30 + t2v * 0.30 + t3v * 0.40
                if acum < 4.0:
                    n_bajo += 1
    materias_en_bajo.append(n_bajo)

df_pred['n_bajo_acumulada'] = materias_en_bajo
df_pred['t3_confirma_riesgo'] = df_pred['n_bajo_acumulada'] >= 1
df_pred['pred_label']    = pred_labels
df_pred['proba_critical'] = proba_critical_predict
df_pred['confianza']      = confianza

print(f"\n── Distribución T3 parcial ──")
print(f"  Con ≥1 materia en bajo: {df_pred['t3_confirma_riesgo'].sum()} estudiantes")
print(f"  Sin materias en bajo:   {(~df_pred['t3_confirma_riesgo']).sum()} estudiantes")

# ── PASO 3: Asignar categorías ──
print("\n" + "=" * 60)
print("PASO 3 — Categorías de alerta (4 cuadrantes)")
print("=" * 60)

def asignar_categoria(row):
    modelo_riesgo = row['pred_label'] in ['critical', 'recovery']
    t3_riesgo     = row['t3_confirma_riesgo']

    if modelo_riesgo and t3_riesgo:
        return 'Riesgo Confirmado'
    elif modelo_riesgo and not t3_riesgo:
        return 'Riesgo Teórico'
    elif not modelo_riesgo and t3_riesgo:
        return 'Punto Ciego'
    else:
        return 'Sin Riesgo'

df_pred['categoria'] = df_pred.apply(asignar_categoria, axis=1)

# Orden de urgencia
orden_cat = {
    'Riesgo Confirmado': 0,
    'Punto Ciego':       1,
    'Riesgo Teórico':    2,
    'Sin Riesgo':        3
}
colores_cat = {
    'Riesgo Confirmado': '#A32D2D',
    'Punto Ciego':       '#BA7517',
    'Riesgo Teórico':    '#185FA5',
    'Sin Riesgo':        '#639922'
}
emojis_cat = {
    'Riesgo Confirmado': '🔴',
    'Punto Ciego':       '🟠',
    'Riesgo Teórico':    '🔵',
    'Sin Riesgo':        '🟢'
}

print(f"\n── Distribución de categorías ──")
for cat in ['Riesgo Confirmado', 'Punto Ciego',
            'Riesgo Teórico', 'Sin Riesgo']:
    n   = (df_pred['categoria'] == cat).sum()
    pct = n / len(df_pred) * 100
    print(f"  {emojis_cat[cat]} {cat:<20}: {n:>3} estudiantes ({pct:.1f}%)")

print(f"\n── Distribución por grado ──")
pivot = df_pred.groupby(['grade', 'categoria']).size().unstack(fill_value=0)
print(pivot.to_string())

# ── PASO 4: Visualizaciones ──
print("\n" + "=" * 60)
print("PASO 4 — Visualizaciones")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(
    'Sistema de Alerta Temprana — Vermont School 2025-26\n'
    f'Random Forest (umbral={UMBRAL_PRODUCCION}) | '
    f'149 estudiantes',
    fontsize=13, fontweight='bold'
)

# ── Plot 1: Scatter cuadrantes ──
ax = axes[0, 0]
for cat in ['Riesgo Confirmado', 'Punto Ciego',
            'Riesgo Teórico', 'Sin Riesgo']:
    mask = df_pred['categoria'] == cat
    ax.scatter(
        df_pred.loc[mask, 'proba_critical'],
        df_pred.loc[mask, 'n_bajo_acumulada'],
        color=colores_cat[cat],
        alpha=0.85 if cat != 'Sin Riesgo' else 0.35,
        s=60 if cat != 'Sin Riesgo' else 30,
        label=f"{emojis_cat[cat]} {cat} "
              f"(n={mask.sum()})",
        zorder=4 if cat != 'Sin Riesgo' else 2
    )

ax.axvline(x=UMBRAL_PRODUCCION, color='gray',
           linestyle='--', linewidth=1, alpha=0.6)
ax.axhline(y=0.5, color='gray',
           linestyle='--', linewidth=1, alpha=0.6)
ax.set_xlabel('Probabilidad de riesgo (modelo)')
ax.set_ylabel('Materias en bajo — acumulada T3 parcial')
ax.set_title('Clasificación de estudiantes — 4 cuadrantes')
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.2)

# ── Plot 2: Distribución por categoría ──
ax = axes[0, 1]
cats    = ['Riesgo Confirmado', 'Punto Ciego',
           'Riesgo Teórico', 'Sin Riesgo']
counts  = [( df_pred['categoria'] == c).sum() for c in cats]
colors  = [colores_cat[c] for c in cats]
bars    = ax.barh(cats, counts, color=colors, alpha=0.85)
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{count} ({count/len(df_pred)*100:.0f}%)',
            va='center', fontsize=10)
ax.set_xlabel('Número de estudiantes')
ax.set_title('Distribución de categorías\nVermont School 2025-26')
ax.grid(alpha=0.3, axis='x')
ax.set_xlim(0, max(counts) * 1.25)

# ── Plot 3: Distribución por grado ──
ax = axes[1, 0]
grados = sorted(df_pred['grade'].unique())
x      = np.arange(len(grados))
width  = 0.2
for i, cat in enumerate(cats):
    vals = [
        (df_pred[df_pred['grade'] == g]['categoria'] == cat).sum()
        for g in grados
    ]
    ax.bar(x + i*width, vals, width,
           label=f"{emojis_cat[cat]} {cat}",
           color=colores_cat[cat], alpha=0.85)
ax.set_xlabel('Grado')
ax.set_ylabel('Estudiantes')
ax.set_title('Categorías por grado')
ax.set_xticks(x + width*1.5)
ax.set_xticklabels([f'{g}°' for g in grados])
ax.legend(fontsize=8)
ax.grid(alpha=0.3, axis='y')

# ── Plot 4: Curva de ganancia acumulada ──
ax = axes[1, 1]
orden_gain  = np.argsort(proba_critical_predict)[::-1]
y_gain      = df_pred.iloc[orden_gain]['t3_confirma_riesgo'].values.astype(int)
n_total     = len(y_gain)
n_riesgo    = y_gain.sum()

ganancia_modelo   = np.cumsum(y_gain) / n_riesgo * 100
pct_revisados     = np.arange(1, n_total + 1) / n_total * 100
ganancia_aleatoria = pct_revisados

ax.plot(pct_revisados, ganancia_modelo,
        color='#A32D2D', linewidth=2.5,
        label='Modelo RF')
ax.plot(pct_revisados, ganancia_aleatoria,
        color='gray', linewidth=1.5, linestyle=':',
        label='Aleatorio')
ax.fill_between(pct_revisados, ganancia_modelo,
                ganancia_aleatoria,
                alpha=0.1, color='#A32D2D')

# Marcar puntos operativos
for pct_rev, color in [(20, '#185FA5'), (40, '#BA7517')]:
    idx_p    = int(pct_rev / 100 * n_total) - 1
    gain_p   = ganancia_modelo[idx_p]
    ax.scatter(pct_rev, gain_p, color=color, s=80, zorder=5)
    ax.annotate(
        f'{gain_p:.0f}% estudiantes\nen riesgo\nrevisando {pct_rev}%',
        (pct_rev, gain_p),
        textcoords="offset points",
        xytext=(8, -30), fontsize=8, color=color
    )

ax.set_xlabel('% estudiantes revisados (por probabilidad)')
ax.set_ylabel('% estudiantes en riesgo detectados')
ax.set_title('Curva de ganancia acumulada\n¿Cuántos revisar para capturar qué % en riesgo?')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(0, 100)
ax.set_ylim(0, 105)

plt.tight_layout()
display(fig)
plt.close()

# ── PASO 5: Feature importance ──
print("\n" + "=" * 60)
print("PASO 5 — Feature importance del modelo final")
print("=" * 60)

importances = pd.DataFrame({
    'feature':    X.columns,
    'importance': rf_final.feature_importances_
}).sort_values('importance', ascending=False)

print("── Top 15 variables más predictivas ──")
print(importances.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 7))
top15    = importances.head(15)
colores_fi = []
for f in top15['feature']:
    if '_T2' in f:      colores_fi.append('#A32D2D')
    elif '_T1' in f:    colores_fi.append('#BA7517')
    elif 'delta' in f or 'tendencia' in f:
                        colores_fi.append('#185FA5')
    else:               colores_fi.append('#639922')

ax.barh(top15['feature'][::-1], top15['importance'][::-1],
        color=colores_fi[::-1], alpha=0.85)
ax.set_xlabel('Importancia')
ax.set_title(
    'Feature Importance — Random Forest Final\n'
    'Vermont EWS | Entrenado 24-25 | T1+T2+features derivadas',
    fontsize=11, fontweight='bold'
)
from matplotlib.patches import Patch
leyenda = [
    Patch(color='#A32D2D', label='Notas T2'),
    Patch(color='#BA7517', label='Notas T1'),
    Patch(color='#185FA5', label='Deltas / Tendencia'),
    Patch(color='#639922', label='Asistencia / Convivencia'),
]
ax.legend(handles=leyenda, fontsize=9)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
display(fig)
plt.close()

print(f"\n── Interpretación pedagógica ──")
for _, row in importances.head(5).iterrows():
    print(f"  {row['feature']:<30} importancia: {row['importance']:.4f}")

# ── PASO 6: Guardar en Silver ──
print("\n" + "=" * 60)
print("PASO 6 — Guardar predicciones en Silver")
print("=" * 60)

df_output = df_pred[[
    'student_id', 'grade', 'section_anon',
    'pred_label', 'proba_critical', 'confianza',
    'n_bajo_acumulada', 't3_confirma_riesgo', 'categoria'
]].copy()
df_output['umbral_usado']    = UMBRAL_PRODUCCION
df_output['modelo']          = 'Random Forest'
df_output['fecha_ejecucion'] = pd.Timestamp.now().strftime('%Y-%m-%d')

spark.createDataFrame(df_output).write\
    .mode("overwrite")\
    .parquet(f"{SILVER}/predictions_25_26_v2")

print(f"  ✓ Predicciones guardadas: {SILVER}/predictions_25_26_v2")
print(f"  ✓ Estudiantes: {len(df_output)}")
print(f"  ✓ Columnas: {list(df_output.columns)}")

print(f"\n{'='*60}")
print(f"CELDA 6 COMPLETA")
print(f"{'='*60}")
print(f"  Modelo:    Random Forest optimizado")
print(f"  Umbral:    {UMBRAL_PRODUCCION}")
print(f"  Estudiantes predichos: {len(df_output)}")
for cat in ['Riesgo Confirmado', 'Punto Ciego',
            'Riesgo Teórico', 'Sin Riesgo']:
    n = (df_output['categoria'] == cat).sum()
    print(f"  {emojis_cat[cat]} {cat}: {n}")
print(f"\n  df_pred disponible para Celda 7 (clustering)")

In [0]:
# CELDA 7 — CLUSTERING K-MEANS
# Perfiles naturales de estudiantes
#
# OBJETIVO: no clasificar riesgo (eso ya lo hace el clasificador)
# sino identificar TIPOS de estudiantes con patrones similares
# para diseñar intervenciones diferenciadas
#
# FEATURES PARA CLUSTERING:
# Variables que describen el perfil integral del estudiante
# (notas, asistencia, convivencia, tendencia)
# Usamos datos de 25-26 (población actual)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

print("=" * 60)
print("CLUSTERING K-MEANS — Perfiles de estudiantes")
print("Vermont School 2025-26 | 149 estudiantes")
print("=" * 60)

# ── Features para clustering ──
# Seleccionamos variables que describen el perfil
# NO incluimos la predicción del modelo (sería circular)
FEATURES_CLUSTER = [
    'avg_T1', 'avg_T2',               # rendimiento general
    'tendencia_general',               # mejorando o deteriorando
    'n_bajo_T1', 'n_bajo_T2',         # materias en bajo
    'dispersion_T2',                   # consistencia entre materias
    'min_nota_T2',                     # materia más débil
    'n_destacadas_T2',                 # materias sobresalientes
    'total_absences',                  # ausencias totales
    'ratio_ausencia_clase',            # patrón de ausencias
    'n_f1', 'n_f2',                    # convivencia
    'indice_disciplinario',            # índice ponderado
]

available_cluster = [f for f in FEATURES_CLUSTER
                     if f in df_predict.columns]
missing_cluster   = [f for f in FEATURES_CLUSTER
                     if f not in df_predict.columns]

print(f"\n✓ Features disponibles: {len(available_cluster)}")
if missing_cluster:
    print(f"  Faltantes: {missing_cluster}")

# Preparar matriz
X_cluster = df_predict[available_cluster].copy()
X_cluster = X_cluster.fillna(X_cluster.mean())

# Estandarizar — OBLIGATORIO para K-Means
# K-Means usa distancias euclidianas
# sin estandarizar, variables con mayor escala dominan
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_cluster)

print(f"✓ Features estandarizadas (media=0, std=1)")
print(f"✓ Shape: {X_scaled.shape}")

# ── PASO 1: Elbow Method ──
print("\n" + "=" * 60)
print("PASO 1 — Elbow Method: número óptimo de clusters")
print("=" * 60)
print("Minimizar inercia (suma de distancias al centroide)")

inercias    = []
silhouettes = []
K_range     = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42,
                n_init=10, max_iter=300)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    silhouettes.append(sil)
    print(f"  K={k}: inercia={km.inertia_:.1f} | "
          f"silhouette={sil:.3f}")

# ── Visualización Elbow ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Elbow Method — Selección de K\nVermont EWS Clustering',
             fontsize=12, fontweight='bold')

ax = axes[0]
ax.plot(list(K_range), inercias, 'o-',
        color='#A32D2D', linewidth=2)
ax.set_xlabel('Número de clusters (K)')
ax.set_ylabel('Inercia')
ax.set_title('Elbow: inercia vs K')
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(list(K_range), silhouettes, 's-',
        color='#185FA5', linewidth=2)
ax.set_xlabel('Número de clusters (K)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette: cohesión vs K\n(mayor = mejor)')
ax.grid(alpha=0.3)

plt.tight_layout()
display(fig)
plt.close()

# ── PASO 2: K óptimo ──
K_OPTIMO = list(K_range)[silhouettes.index(max(silhouettes))]
print(f"\n── K óptimo según Silhouette: {K_OPTIMO} ──")
print(f"  Silhouette score: {max(silhouettes):.3f}")
print(f"  (confirmado visualmente con Elbow)")

# ── PASO 3: K-Means final ──
print("\n" + "=" * 60)
print(f"PASO 2 — K-Means con K={K_OPTIMO}")
print("=" * 60)

km_final = KMeans(n_clusters=K_OPTIMO, random_state=42,
                   n_init=10, max_iter=300)
clusters = km_final.fit_predict(X_scaled)
df_pred['cluster'] = clusters

print(f"✓ Clustering completo")
print(f"\n── Distribución de clusters ──")
for k in range(K_OPTIMO):
    n   = (clusters == k).sum()
    pct = n / len(clusters) * 100
    print(f"  Cluster {k}: {n:>3} estudiantes ({pct:.1f}%)")

# ── PASO 4: Caracterizar perfiles ──
print("\n" + "=" * 60)
print("PASO 3 — Caracterización de perfiles")
print("=" * 60)

# Media de cada feature por cluster
perfil = df_pred.groupby('cluster')[available_cluster].mean().round(3)
print("\n── Medias por cluster ──")
print(perfil.T.to_string())

# Cruzar con categorías de alerta
print("\n── Categorías de alerta por cluster ──")
cross = pd.crosstab(df_pred['cluster'], df_pred['categoria'])
cross['Total'] = cross.sum(axis=1)
print(cross.to_string())

# Cruzar con grado
print("\n── Grado por cluster ──")
cross_grade = pd.crosstab(df_pred['cluster'],
                           df_pred['grade'])
cross_grade['Total'] = cross_grade.sum(axis=1)
print(cross_grade.to_string())

# ── PASO 5: Nombrar perfiles ──
print("\n" + "=" * 60)
print("PASO 4 — Interpretación pedagógica de perfiles")
print("=" * 60)
print("(Los nombres se asignan según las características")
print(" de cada cluster — ajustar según resultados reales)\n")

# Calcular percentiles para clasificar cada cluster
avg_global   = df_pred['avg_T2'].mean()
aus_global   = df_pred['total_absences'].mean()
disc_global  = df_pred['indice_disciplinario'].mean()
tend_global  = df_pred['tendencia_general'].mean()
dest_global  = df_pred['n_destacadas_T2'].mean()

nombres_cluster = {}
for k in range(K_OPTIMO):
    mask   = df_pred['cluster'] == k
    sub    = df_pred[mask]
    avg_k  = sub['avg_T2'].mean()
    aus_k  = sub['total_absences'].mean()
    disc_k = sub['indice_disciplinario'].mean()
    tend_k = sub['tendencia_general'].mean()
    dest_k = sub['n_destacadas_T2'].mean()
    bajo_k = sub['n_bajo_T2'].mean()

    print(f"  Cluster {k} ({mask.sum()} estudiantes):")
    print(f"    avg_T2:     {avg_k:.2f} (global: {avg_global:.2f})")
    print(f"    ausencias:  {aus_k:.1f} (global: {aus_global:.1f})")
    print(f"    disciplina: {disc_k:.2f} (global: {disc_global:.2f})")
    print(f"    tendencia:  {tend_k:.3f} (global: {tend_global:.3f})")
    print(f"    destacadas: {dest_k:.2f} (global: {dest_global:.2f})")
    print(f"    n_bajo_T2:  {bajo_k:.2f}")

    # Asignar nombre automático según perfil
    if avg_k > avg_global * 1.1 and dest_k > dest_global:
        nombre = "Alto rendimiento"
    elif avg_k > avg_global and bajo_k < 0.5:
        nombre = "Rendimiento sólido"
    elif bajo_k >= 2 and aus_k > aus_global:
        nombre = "Riesgo alto — desvinculado"
    elif bajo_k >= 1 and disc_k > disc_global:
        nombre = "Riesgo conductual"
    elif bajo_k >= 1 and aus_k <= aus_global:
        nombre = "Dificultad académica"
    elif tend_k < tend_global - 0.1:
        nombre = "En deterioro"
    else:
        nombre = "Promedio estable"

    nombres_cluster[k] = nombre
    print(f"    → Perfil: {nombre}\n")

df_pred['perfil'] = df_pred['cluster'].map(nombres_cluster)

# ── PASO 6: Visualizaciones ──
print("=" * 60)
print("PASO 5 — Visualizaciones de clustering")
print("=" * 60)

colores_cluster = ['#A32D2D', '#185FA5', '#639922',
                    '#BA7517', '#534AB7', '#0F6E56'][:K_OPTIMO]

# ── PCA para visualizar en 2D ──
pca     = PCA(n_components=2, random_state=42)
X_pca   = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    f'K-Means Clustering (K={K_OPTIMO}) — Perfiles de estudiantes\n'
    f'Vermont School 2025-26',
    fontsize=12, fontweight='bold'
)

# Plot 1: PCA con clusters
ax = axes[0]
for k in range(K_OPTIMO):
    mask = clusters == k
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               color=colores_cluster[k], s=60, alpha=0.8,
               label=f'C{k}: {nombres_cluster[k]} (n={mask.sum()})')
ax.set_xlabel(f'PC1 ({var_exp[0]*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({var_exp[1]*100:.1f}% varianza)')
ax.set_title('Proyección PCA — clusters en 2D')
ax.legend(fontsize=8, loc='best')
ax.grid(alpha=0.3)

# Plot 2: Heatmap de perfiles
ax = axes[1]
# Normalizar medias para comparar en la misma escala
perfil_norm = (perfil - perfil.mean()) / (perfil.std() + 1e-8)
im = ax.imshow(perfil_norm.T, cmap='RdYlGn',
               aspect='auto', vmin=-2, vmax=2)
ax.set_xticks(range(K_OPTIMO))
ax.set_xticklabels([f'C{k}\n{nombres_cluster[k][:10]}'
                     for k in range(K_OPTIMO)], fontsize=8)
ax.set_yticks(range(len(available_cluster)))
ax.set_yticklabels(available_cluster, fontsize=8)
ax.set_title('Heatmap de perfiles\n(verde=alto, rojo=bajo vs. media global)')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
display(fig)
plt.close()

# ── Radar chart por perfil ──
from matplotlib.patches import FancyArrowPatch

# Variables para el radar (las más interpretables)
radar_vars = ['avg_T2', 'tendencia_general', 'n_destacadas_T2',
              'ratio_ausencia_clase', 'indice_disciplinario',
              'min_nota_T2']
radar_labels = ['Promedio T2', 'Tendencia',
                'Destacadas', 'Ausencias',
                'Disciplina', 'Nota mínima T2']

N    = len(radar_vars)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, K_OPTIMO,
                          figsize=(4*K_OPTIMO, 4),
                          subplot_kw=dict(polar=True))
fig.suptitle(
    f'Radar de perfiles — K={K_OPTIMO} clusters\n'
    f'Vermont School 2025-26',
    fontsize=12, fontweight='bold'
)

if K_OPTIMO == 1:
    axes = [axes]

# Normalizar cada variable a 0-1 para el radar
for k, ax in enumerate(axes):
    vals = []
    for v in radar_vars:
        col_min = df_pred[v].min()
        col_max = df_pred[v].max()
        cluster_mean = df_pred[df_pred['cluster'] == k][v].mean()
        if col_max > col_min:
            norm = (cluster_mean - col_min) / (col_max - col_min)
        else:
            norm = 0.5
        # Invertir ausencias y disciplina (menos es mejor)
        if v in ['ratio_ausencia_clase', 'indice_disciplinario']:
            norm = 1 - norm
        vals.append(norm)
    vals += vals[:1]

    ax.plot(angles, vals, color=colores_cluster[k],
            linewidth=2)
    ax.fill(angles, vals, color=colores_cluster[k],
            alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f'C{k}: {nombres_cluster[k]}\n'
                 f'(n={(clusters==k).sum()})',
                 fontsize=9, fontweight='bold',
                 color=colores_cluster[k])
    ax.grid(alpha=0.3)

plt.tight_layout()
display(fig)
plt.close()

# ── PASO 7: Guardar ──
print("\n" + "=" * 60)
print("PASO 6 — Guardar clusters en Silver")
print("=" * 60)

df_clusters_out = df_pred[[
    'student_id', 'grade', 'section_anon',
    'cluster', 'perfil', 'categoria',
    'avg_T1', 'avg_T2', 'tendencia_general',
    'n_bajo_T1', 'n_bajo_T2', 'total_absences',
    'n_f1', 'n_f2', 'indice_disciplinario',
    'n_destacadas_T2', 'min_nota_T2'
]].copy()

spark.createDataFrame(df_clusters_out).write\
    .mode("overwrite")\
    .parquet(f"{SILVER}/clusters_25_26")

print(f"  ✓ Guardado: {SILVER}/clusters_25_26")
print(f"  ✓ Estudiantes: {len(df_clusters_out)}")

print(f"\n{'='*60}")
print(f"CELDA 7 COMPLETA — K-Means clustering")
print(f"{'='*60}")
print(f"  K óptimo: {K_OPTIMO}")
print(f"  Silhouette score: {max(silhouettes):.3f}")
for k in range(K_OPTIMO):
    n = (clusters == k).sum()
    print(f"  C{k} — {nombres_cluster[k]}: {n} estudiantes")
print(f"\n  df_pred con cluster y perfil disponible")
print(f"  Listo para Celda 8 — persistencia de modelos")

In [0]:
# CELDA 7b — Corregir nombre Cluster 0

# Actualizar nombre
nombres_cluster[0] = "Riesgo multidimensional"
df_pred['perfil']  = df_pred['cluster'].map(nombres_cluster)

print("── Perfiles actualizados ──")
for k in range(K_OPTIMO):
    n = (df_pred['cluster'] == k).sum()
    print(f"  C{k} — {nombres_cluster[k]}: {n} estudiantes")

# Guardar de nuevo con nombre corregido
df_clusters_out = df_pred[[
    'student_id', 'grade', 'section_anon',
    'cluster', 'perfil', 'categoria',
    'avg_T1', 'avg_T2', 'tendencia_general',
    'n_bajo_T1', 'n_bajo_T2', 'total_absences',
    'n_f1', 'n_f2', 'indice_disciplinario',
    'n_destacadas_T2', 'min_nota_T2'
]].copy()

spark.createDataFrame(df_clusters_out).write \
    .mode("overwrite") \
    .parquet(f"{SILVER}/clusters_25_26")

print(f"\n✓ Clusters guardados con nombre corregido")

In [0]:
# CELDA 8 — PERSISTENCIA DE MODELOS (versión corregida)
# Guarda directamente en Databricks Volumes sin pasar por /tmp

import joblib
import json
import os
import tempfile
from datetime import datetime

print("=" * 60)
print("PERSISTENCIA DE MODELOS — Vermont EWS")
print("=" * 60)

VOLUME_MODEL_DIR = f"{SILVER}/models"
dbutils.fs.mkdirs(VOLUME_MODEL_DIR)

# En Databricks, los Volumes se acceden como filesystem normal
# La ruta /Volumes/... es accesible directamente con joblib
VOLUME_PATH = "/Volumes/workspace/vermont/silver/models"
os.makedirs(VOLUME_PATH, exist_ok=True)

# ── 1. Modelo RF ──
print("\n── 1. Modelo Random Forest ──")
model_path = f"{VOLUME_PATH}/rf_classifier.joblib"
joblib.dump(rf_final, model_path)
size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f"  ✓ rf_classifier.joblib ({size_mb:.2f} MB)")

# ── 2. Imputador ──
print("\n── 2. Imputador ──")
joblib.dump(imputer, f"{VOLUME_PATH}/imputer.joblib")
print(f"  ✓ imputer.joblib")

# ── 3. Label Encoder ──
print("\n── 3. Label Encoder ──")
joblib.dump(le, f"{VOLUME_PATH}/label_encoder.joblib")
print(f"  ✓ label_encoder.joblib")
print(f"  → {dict(zip(range(len(classes)), classes))}")

# ── 4. Features ──
print("\n── 4. Lista de features ──")
with open(f"{VOLUME_PATH}/feature_list.json", 'w') as f:
    json.dump(list(X.columns), f, indent=2)
print(f"  ✓ feature_list.json ({len(X.columns)} features)")

# ── 5. Metadatos ──
print("\n── 5. Metadatos ──")
metadata = {
    "modelo":              "RandomForestClassifier",
    "version":             "v2",
    "fecha_entrenamiento": datetime.now().strftime("%Y-%m-%d"),
    "dataset_train":       "Vermont School 2024-25",
    "n_estudiantes_train": int(len(X)),
    "n_features":          int(len(X.columns)),
    "hiperparametros":     grid_search.best_params_,
    "umbral_produccion":   UMBRAL_PRODUCCION,
    "clases":              list(classes),
    "metricas_cv": {
        "f1_macro":        round(float(f1_final), 4),
        "recall_critical": round(float(recall_final), 4),
        "f2_critical":     round(float(f2_final), 4),
    },
    "justificacion_umbral": (
        f"Umbral {UMBRAL_PRODUCCION} maximiza F1 macro (0.663) "
        f"con Recall critical 0.833, detectando 15 de 18 "
        f"críticos con 11 falsas alarmas."
    ),
    "anti_leakage": [
        "Modelo entrenado solo con T1+T2",
        "T3 NO usado como feature",
        "Imputador ajustado solo con train",
        "CV estratificada para evaluación honesta"
    ]
}
with open(f"{VOLUME_PATH}/model_metadata.json", 'w',
          encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"  ✓ model_metadata.json")

# ── 6. Verificar carga ──
print("\n── 6. Verificación de carga ──")
rf_loaded  = joblib.load(f"{VOLUME_PATH}/rf_classifier.joblib")
imp_loaded = joblib.load(f"{VOLUME_PATH}/imputer.joblib")
le_loaded  = joblib.load(f"{VOLUME_PATH}/label_encoder.joblib")
with open(f"{VOLUME_PATH}/feature_list.json") as f:
    features_loaded = json.load(f)

# Test rápido
X_test       = pd.DataFrame(
    imp_loaded.transform(
        df_predict[features_loaded].fillna(0)
    ),
    columns=features_loaded
)
y_test_proba = rf_loaded.predict_proba(X_test)
print(f"  ✓ Modelo cargado y funcionando")
print(f"  ✓ Predicción de prueba: {y_test_proba.shape}")
print(f"  ✓ Features verificadas: {len(features_loaded)}")

# ── 7. Listar archivos guardados ──
print("\n── 7. Archivos en Volumes ──")
files = dbutils.fs.ls(VOLUME_MODEL_DIR)
for f in files:
    print(f"  ✓ {f.name} ({f.size/1024:.1f} KB)")

print(f"\n── Para cargar en Streamlit ──")
print(f"""
  import joblib, json
  rf    = joblib.load('{VOLUME_PATH}/rf_classifier.joblib')
  imp   = joblib.load('{VOLUME_PATH}/imputer.joblib')
  le    = joblib.load('{VOLUME_PATH}/label_encoder.joblib')
  feats = json.load(open('{VOLUME_PATH}/feature_list.json'))

  proba  = rf.predict_proba(imp.transform(df[feats]))[:, 0]
  alerta = (proba >= {UMBRAL_PRODUCCION}).astype(int)
""")

print(f"\n{'='*60}")
print(f"CELDA 8 COMPLETA ✓")
print(f"{'='*60}")
print(f"  Ubicación: {VOLUME_PATH}/")
print(f"  Archivos guardados: 5")
print(f"\n  NOTEBOOK 03v2 COMPLETO ✓")
print(f"  Clasificación → Optimización → Clustering → Persistencia")
print(f"  Siguiente sesión: Celda 9 — Regresión por materia")